# Volume Equations: INFYS and Custom Columns

This tutorial demonstrates the volume-equation side of the same `miaproc.biomass` API. The packaged `infys` slice currently uses `response_variable == "VRTAcc"` for accumulated total roll volume.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name != "mia-tutorials" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DATA = ROOT / "data"
REPOS = ROOT.parent
ROOT

import pandas as pd
from miaproc.biomass import BiomassColumns, estimate_trees, load_packaged_equations

equations = load_packaged_equations()
infys = equations[equations["source_dataset"].eq("infys")].copy()
infys.shape


## Pick one concrete equation

For a tutorial row, choose one equation that has species, state, DBH range, height range, and a `VRTAcc` response.


In [ ]:
example_eq = infys.dropna(subset=["scientific_name_apg_raw", "state", "equation_numpy"]).iloc[0]
example_eq[[
    "source_record_id",
    "scientific_name_apg_raw",
    "state",
    "assignment_level",
    "response_variable",
    "response_units",
    "dbh_min_cm",
    "dbh_max_cm",
    "height_min_m",
    "height_max_m",
    "equation_numpy",
]]


## Build a tiny custom-column table

This table uses field labels that are different from the package defaults, so `BiomassColumns` keeps the tutorial close to real field data.


In [ ]:
field_rows = pd.DataFrame({
    "Species": [example_eq["scientific_name_apg_raw"], example_eq["scientific_name_apg_raw"]],
    "DBH (cm)": [10.0, 10.0],
    "Total Height (m)": [8.0, None],
    "LifeStage": ["Adult", "Adult"],
})
field_rows


In [ ]:
cols = BiomassColumns(
    species="Species",
    dbh_cm="DBH (cm)",
    height_m="Total Height (m)",
    life_stage="LifeStage",
)

out = estimate_trees(
    field_rows,
    equations=equations,
    columns=cols,
    state=example_eq["state"],
    dataset="infys",
    response_variable=example_eq["response_variable"],
)
out[[
    "Species",
    "DBH (cm)",
    "Total Height (m)",
    "estimate_response_variable",
    "response_variable",
    "match_status",
    "range_status",
    "source_record_id",
    "equation_code",
]]


## What this teaches

The first row is evaluated. The second row matches an equation but cannot be evaluated because the volume expression requires height, so `match_status` reports `height_missing`.


In [ ]:
out[["match_status", "range_status"]].value_counts(dropna=False).rename("rows")
